## Gmail API (OAuth) — fetch your emails (tutorial)

This notebook shows how to connect to **Gmail** using **OAuth 2.0** and fetch your recent emails via the **Gmail API**.

### Before you run

- Create a Google Cloud project
- Enable **Gmail API**
- Configure **OAuth consent screen**
- Create an **OAuth Client ID** (Desktop app is simplest for notebooks)
- Download the client secrets JSON and save it as `credentials.json` next to this notebook

### Files created/used

- `.env` (already created): points to `credentials.json`, `token.json`, and scopes
- `credentials.json`: OAuth client secrets (you download this)
- `token.json`: cached OAuth tokens (created by this notebook after login)

`credentials.json`, `token.json`, and `.env` should stay **local** (they’re gitignored).

In [41]:
# If you're in a fresh environment, install dependencies:
# (You can also: pip install -r requirements.txt)

%pip -q install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [42]:
import os
import json
from pathlib import Path

from dotenv import load_dotenv

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

load_dotenv()

CLIENT_SECRETS_FILE = os.getenv("GOOGLE_CLIENT_SECRETS_FILE", "credentials.json")
TOKEN_FILE = os.getenv("GMAIL_TOKEN_FILE", "token.json")
SCOPES = os.getenv("GMAIL_SCOPES", "https://www.googleapis.com/auth/gmail.readonly").split()

print("CLIENT_SECRETS_FILE:", CLIENT_SECRETS_FILE)
print("TOKEN_FILE:", TOKEN_FILE)
print("SCOPES:", SCOPES)

if not Path(CLIENT_SECRETS_FILE).exists():
    raise FileNotFoundError(
        f"Missing {CLIENT_SECRETS_FILE}. Download OAuth client secrets JSON from Google Cloud Console "
        f"and save it next to this notebook (or update GOOGLE_CLIENT_SECRETS_FILE in .env)."
    )

CLIENT_SECRETS_FILE: credentials.json
TOKEN_FILE: token.json
SCOPES: ['https://www.googleapis.com/auth/gmail.readonly']


In [43]:
def get_gmail_service(*, client_secrets_file: str, token_file: str, scopes: list[str]):
    """Returns an authenticated Gmail API service. Creates/refreshes token_file as needed."""
    creds = None
    token_path = Path(token_file)

    if token_path.exists():
        creds = Credentials.from_authorized_user_file(token_file, scopes)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(client_secrets_file, scopes)
            # This opens a browser window to let you sign in and approve.
            # port=0 picks an available local port automatically.
            creds = flow.run_local_server(port=0)

        token_path.write_text(creds.to_json(), encoding="utf-8")

    return build("gmail", "v1", credentials=creds)


service = get_gmail_service(
    client_secrets_file=CLIENT_SECRETS_FILE,
    token_file=TOKEN_FILE,
    scopes=SCOPES,
)

print("Gmail service created.")

Gmail service created.


In [44]:
def _header(headers: list[dict], name: str) -> str | None:
    name_lower = name.lower()
    for h in headers or []:
        if (h.get("name") or "").lower() == name_lower:
            return h.get("value")
    return None


def fetch_recent_messages(max_results: int = 5, query: str | None = None):
    """Fetches recent messages and prints basic metadata."""
    try:
        resp = service.users().messages().list(
            userId="me",
            maxResults=max_results,
            q=query,
        ).execute()

        msgs = resp.get("messages", [])
        if not msgs:
            print("No messages found.")
            return

        for i, m in enumerate(msgs, start=1):
            msg = service.users().messages().get(
                userId="me",
                id=m["id"],
                format="metadata",
                metadataHeaders=["From", "To", "Subject", "Date"],
            ).execute()

            payload = msg.get("payload", {})
            headers = payload.get("headers", [])

            frm = _header(headers, "From")
            subj = _header(headers, "Subject")
            date = _header(headers, "Date")
            snippet = (msg.get("snippet") or "").replace("\n", " ")

            print(f"\n{i}. {subj or '(no subject)'}")
            print(f"   From: {frm}")
            print(f"   Date: {date}")
            print(f"   Snippet: {snippet}")

    except HttpError as e:
        # Common causes: wrong OAuth client type, wrong redirect URIs, missing Gmail API enablement,
        # or scope mismatch.
        raise RuntimeError(f"Gmail API error: {e}") from e


# Fetch the 5 most recent emails in your inbox
fetch_recent_messages(max_results=50, query="in:inbox (category:primary OR category:updates)")


1. Royal Bank of Canada is hiring for Data Scientist + 28 new data scientist jobs in Toronto, ON
   From: Indeed <donotreply@jobalert.indeed.com>
   Date: Fri, 08 May 2026 00:06:37 +0000
   Snippet: Apply to jobs at Royal Bank of Canada, BMO Financial Group and Amazon Development Centre Canada ULC ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌ ‌

2. Top AI Demos #25: Voice Agents, MLX Trading Cards & Cyber Threat Intel
   From: AI Tinkerers - Post-Training <post-training@mail.aitinkerers.org>
   Date: Thu, 7 May 2026 22:44:02 +0000
   Snippet: Top AI Demos #25: Voice Agents, MLX Trading Cards &amp; Cyber Threat Intel Issue #25 · Week of May 4 Joe Heitzeberg LinkedIn X/Twitter GitHub • Founder at AI Tinkerers • ⏱️ 1 min read Creating space

3. Your OpenAI API account has been funded
   From: noreply@tm.openai.com
   Date: Thu, 07 May 2026 22:11:20 +0000 (UTC)
   Snippet: Hi there, We charged $5.65 to your credit card ending in 3668 t

In [45]:
import re
import base64
import sqlite3
from datetime import datetime
from typing import Any, Optional

from bs4 import BeautifulSoup
from dateutil import parser as date_parser

from openai import OpenAI
from dotenv import load_dotenv
import os

# Load environment variables from .env file if present
load_dotenv()

# Accept either env var name
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or os.getenv("openai_api_key")
if not OPENAI_API_KEY:
    raise ValueError(
        "Missing OpenAI API key. Add OPENAI_API_KEY=... to your .env (or openai_api_key=...)."
    )

client = OpenAI(api_key=OPENAI_API_KEY)

DB_PATH = os.getenv("JOBTRACKER_DB", "jobtracker.sqlite3")
MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")

print("DB_PATH:", DB_PATH)
print("MODEL:", MODEL)

DB_PATH: jobtracker.sqlite3
MODEL: gpt-5-nano


In [46]:
def init_db(db_path: str = DB_PATH) -> None:
    conn = sqlite3.connect(db_path)
    try:
        conn.execute("PRAGMA journal_mode=WAL;")
        conn.execute("PRAGMA foreign_keys=ON;")

        # Stores raw email content for the job-related subset we care about.
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS emails (
              gmail_message_id TEXT PRIMARY KEY,
              gmail_thread_id TEXT,
              internal_date_ms INTEGER,
              from_addr TEXT,
              to_addr TEXT,
              subject TEXT,
              date_raw TEXT,
              snippet TEXT,
              body_text TEXT,
              created_at TEXT NOT NULL
            );
            """
        )

        # Records that a message_id has been processed so reruns don't re-call OpenAI.
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS processed_messages (
              gmail_message_id TEXT PRIMARY KEY,
              category TEXT,
              confidence REAL,
              model TEXT,
              raw_json TEXT,
              processed_at TEXT NOT NULL
            );
            """
        )

        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS applications (
              id INTEGER PRIMARY KEY AUTOINCREMENT,
              app_key TEXT NOT NULL UNIQUE,
              company TEXT,
              job_title TEXT,
              job_id TEXT,
              source TEXT,
              applied_date TEXT,
              status TEXT NOT NULL,
              last_update_date TEXT,
              last_email_message_id TEXT,
              last_email_date_raw TEXT,
              confidence REAL,
              notes TEXT,
              created_at TEXT NOT NULL,
              updated_at TEXT NOT NULL
            );
            """
        )

        # Event history (email-derived + manual updates)
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS app_events (
              id INTEGER PRIMARY KEY AUTOINCREMENT,
              app_key TEXT NOT NULL,
              event_type TEXT NOT NULL,
              event_date TEXT,
              gmail_message_id TEXT,
              raw_json TEXT,
              created_at TEXT NOT NULL,
              FOREIGN KEY(app_key) REFERENCES applications(app_key)
            );
            """
        )

        conn.commit()
    finally:
        conn.close()


init_db()
print("DB initialized.")

DB initialized.


In [47]:
def _b64url_decode(data: str) -> bytes:
    # Gmail uses base64url ("-" and "_")
    return base64.urlsafe_b64decode(data.encode("utf-8"))


def _extract_bodies(payload: dict) -> dict[str, str]:
    """Return {'text/plain': ..., 'text/html': ...} where available."""
    out: dict[str, str] = {}

    def walk(part: dict):
        mime = part.get("mimeType")
        body = part.get("body") or {}
        data = body.get("data")
        if data and mime in ("text/plain", "text/html"):
            try:
                out[mime] = _b64url_decode(data).decode("utf-8", errors="replace")
            except Exception:
                out[mime] = ""

        for p in part.get("parts") or []:
            walk(p)

    walk(payload or {})
    return out


def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    # Remove script/style
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text("\n")
    # Normalize whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def get_message_full(message_id: str) -> dict:
    return service.users().messages().get(userId="me", id=message_id, format="full").execute()


def normalize_date(date_raw: Optional[str]) -> Optional[str]:
    if not date_raw:
        return None
    try:
        dt = date_parser.parse(date_raw)
        return dt.isoformat()
    except Exception:
        return None

In [72]:
JOB_EMAIL_SYSTEM_PROMPT = """You are a precise information extraction system.

You will be given a single email about jobs/careers (or not related).
Your task: decide whether it relates to a job application the user made, and if so, classify the event and extract key fields.

Return ONLY valid JSON matching this schema:
{
  "is_job_related": boolean,
  "category": "application_confirmation" | "rejection" | "follow_up" | "interview" | "offer" | "job_alert" | "newsletter" | "other",
  "company": string | null,
  "job_title": string | null,
  "job_id": string | null,
  "applied_date": string | null,        // ISO 8601 if present
  "event_date": string | null,          // ISO 8601 if present (email date is acceptable if nothing else)
  "confidence": number,                // 0..1
  "reason": string,                    // short explanation
  "evidence": {
    "company": string | null,
    "job_title": string | null,
    "job_id": string | null
  },
  "notes": string | null
}

Rules:
- If it is not about a job application process (e.g. grocery promos, receipts), set is_job_related=false.
- If it's about a job posting alert or LinkedIn “add connection” etc, keep is_job_related=true but category="job_alert" or "other".
- Prefer company/job_title/job_id only when clearly supported; otherwise null.
"""


def classify_email_with_openai(*, subject: str | None, from_addr: str | None, date_raw: str | None, snippet: str | None, body_text: str | None) -> dict[str, Any]:
    user_content = {
        "from": from_addr,
        "subject": subject,
        "date": date_raw,
        "snippet": snippet,
        "body": (body_text or "")[:1500],
    }

    resp = client.chat.completions.create(
        model=MODEL,
        # temperature=0,
        messages=[
            {"role": "system", "content": JOB_EMAIL_SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(user_content, ensure_ascii=False)},
        ],
        response_format={"type": "json_object"},
    )

    content = resp.choices[0].message.content or "{}"
    return json.loads(content)

In [73]:
ALLOWED_APP_CATEGORIES = {"application_confirmation", "interview", "rejection"}


def make_app_key(company: Optional[str], job_title: Optional[str], job_id: Optional[str]) -> str:
    # Stable-ish unique key; if job_id exists use it, otherwise fall back to company+title.
    c = (company or "").strip().lower()
    t = (job_title or "").strip().lower()
    j = (job_id or "").strip().lower()
    if j:
        return f"job_id:{j}"
    if c or t:
        return f"company_title:{c}|{t}".strip("|")
    # Worst case: unknown bucket
    return "unknown"


def is_message_processed(db_path: str, gmail_message_id: str) -> bool:
    conn = sqlite3.connect(db_path)
    try:
        row = conn.execute(
            "SELECT 1 FROM processed_messages WHERE gmail_message_id = ? LIMIT 1;",
            (gmail_message_id,),
        ).fetchone()
        return row is not None
    finally:
        conn.close()


def mark_message_processed(db_path: str, gmail_message_id: str, extracted: dict[str, Any]) -> None:
    now = datetime.utcnow().isoformat()
    conn = sqlite3.connect(db_path)
    try:
        conn.execute(
            """
            INSERT INTO processed_messages (gmail_message_id, category, confidence, model, raw_json, processed_at)
            VALUES (?, ?, ?, ?, ?, ?)
            ON CONFLICT(gmail_message_id) DO UPDATE SET
              category=excluded.category,
              confidence=excluded.confidence,
              model=excluded.model,
              raw_json=excluded.raw_json,
              processed_at=excluded.processed_at;
            """,
            (
                gmail_message_id,
                extracted.get("category"),
                extracted.get("confidence"),
                MODEL,
                json.dumps(extracted, ensure_ascii=False),
                now,
            ),
        )
        conn.commit()
    finally:
        conn.close()


def upsert_email_and_event(
    *,
    db_path: str,
    gmail_message_id: str,
    gmail_thread_id: Optional[str],
    internal_date_ms: Optional[int],
    from_addr: Optional[str],
    to_addr: Optional[str],
    subject: Optional[str],
    date_raw: Optional[str],
    snippet: Optional[str],
    body_text: Optional[str],
    extracted: dict[str, Any],
) -> None:
    """Write to SQL ONLY for confirmation/interview/rejection categories."""
    now = datetime.utcnow().isoformat()

    status = extracted.get("category") or "other"
    if status not in ALLOWED_APP_CATEGORIES:
        return

    app_key = make_app_key(extracted.get("company"), extracted.get("job_title"), extracted.get("job_id"))
    conf = extracted.get("confidence")

    conn = sqlite3.connect(db_path)
    try:
        # Store the email (only for allowed categories)
        conn.execute(
            """
            INSERT INTO emails (
              gmail_message_id, gmail_thread_id, internal_date_ms,
              from_addr, to_addr, subject, date_raw, snippet, body_text, created_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(gmail_message_id) DO UPDATE SET
              gmail_thread_id=excluded.gmail_thread_id,
              internal_date_ms=excluded.internal_date_ms,
              from_addr=excluded.from_addr,
              to_addr=excluded.to_addr,
              subject=excluded.subject,
              date_raw=excluded.date_raw,
              snippet=excluded.snippet,
              body_text=excluded.body_text;
            """,
            (
                gmail_message_id,
                gmail_thread_id,
                internal_date_ms,
                from_addr,
                to_addr,
                subject,
                date_raw,
                snippet,
                body_text,
                now,
            ),
        )

        applied_date = extracted.get("applied_date")
        event_date = extracted.get("event_date") or normalize_date(date_raw)

        conn.execute(
            """
            INSERT INTO applications (
              app_key, company, job_title, job_id, source, applied_date,
              status, last_update_date, last_email_message_id, last_email_date_raw,
              confidence, notes, created_at, updated_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(app_key) DO UPDATE SET
              company=COALESCE(excluded.company, applications.company),
              job_title=COALESCE(excluded.job_title, applications.job_title),
              job_id=COALESCE(excluded.job_id, applications.job_id),
              status=excluded.status,
              last_update_date=excluded.last_update_date,
              last_email_message_id=excluded.last_email_message_id,
              last_email_date_raw=excluded.last_email_date_raw,
              confidence=excluded.confidence,
              notes=excluded.notes,
              updated_at=excluded.updated_at;
            """,
            (
                app_key,
                extracted.get("company"),
                extracted.get("job_title"),
                extracted.get("job_id"),
                from_addr,
                applied_date,
                status,
                event_date,
                gmail_message_id,
                date_raw,
                conf,
                extracted.get("notes") or extracted.get("reason"),
                now,
                now,
            ),
        )

        conn.execute(
            """
            INSERT INTO app_events (app_key, event_type, event_date, gmail_message_id, raw_json, created_at)
            VALUES (?, ?, ?, ?, ?, ?);
            """,
            (
                app_key,
                status,
                event_date,
                gmail_message_id,
                json.dumps(extracted, ensure_ascii=False),
                now,
            ),
        )

        conn.commit()
    finally:
        conn.close()

In [84]:
def process_inbox_to_db(max_results: int = 50, query: str = "in:inbox", *, force: bool = False):
    """Fetch messages from Gmail, classify with OpenAI, and store ONLY allowed categories.

    - Rerun-safe: skips message IDs already in processed_messages (unless force=True)
    - Writes to SQL (emails/applications/app_events) only for confirmation/interview/rejection
    """
    resp = service.users().messages().list(userId="me", maxResults=max_results, q=query).execute()
    msgs = resp.get("messages", [])
    print(f"Found {len(msgs)} messages.")

    skipped = 0
    processed = 0
    stored = 0

    for idx, m in enumerate(msgs, start=1):
        message_id = m["id"]

        if not force and is_message_processed(DB_PATH, message_id):
            skipped += 1
            continue

        # Pull full payload so we can extract body
        full = get_message_full(message_id)
        payload = full.get("payload") or {}
        headers = payload.get("headers") or []

        from_addr = _header(headers, "From")
        to_addr = _header(headers, "To")
        subject = _header(headers, "Subject")
        date_raw = _header(headers, "Date")
        snippet = (full.get("snippet") or "").replace("\n", " ")

        bodies = _extract_bodies(payload)
        body_text = bodies.get("text/plain")
        if not body_text and bodies.get("text/html"):
            body_text = html_to_text(bodies["text/html"])

        extracted = classify_email_with_openai(
            subject=subject,
            from_addr=from_addr,
            date_raw=date_raw,
            snippet=snippet,
            body_text=body_text,
        )

        # Always mark as processed so we never re-call OpenAI for this message_id
        mark_message_processed(DB_PATH, message_id, extracted)
        processed += 1

        # Only store application records for allowed categories
        before = extracted.get("category")
        upsert_email_and_event(
            db_path=DB_PATH,
            gmail_message_id=message_id,
            gmail_thread_id=full.get("threadId"),
            internal_date_ms=int(full.get("internalDate")) if full.get("internalDate") else None,
            from_addr=from_addr,
            to_addr=to_addr,
            subject=subject,
            date_raw=date_raw,
            snippet=snippet,
            body_text=body_text,
            extracted=extracted,
        )
        if before in ALLOWED_APP_CATEGORIES:
            stored += 1

        cat = extracted.get("category")
        comp = extracted.get("company")
        title = extracted.get("job_title")
        print(f"[{idx}/{len(msgs)}] {cat} | {comp} | {title} | subj={subject!r}")

    print(f"Done. skipped={skipped} processed_now={processed} stored_app_records={stored}")


# Run this to process your recent inbox messages (rerun-safe)
process_inbox_to_db(max_results=800, query="in:inbox (category:primary OR category:updates)", force=False)

Found 800 messages (after pagination).


KeyboardInterrupt: 

In [75]:
def fetch_applications(status: str | None = None, company_like: str | None = None):
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        sql = "SELECT * FROM applications WHERE 1=1"
        params: list[Any] = []
        if status:
            sql += " AND status = ?"
            params.append(status)
        if company_like:
            sql += " AND company LIKE ?"
            params.append(f"%{company_like}%")
        sql += " ORDER BY updated_at DESC LIMIT 100"

        rows = conn.execute(sql, params).fetchall()
        return [dict(r) for r in rows]
    finally:
        conn.close()


# Example: show the most recent 20 app records
apps = fetch_applications()
apps[:20]

[{'id': 191,
  'app_key': 'company_title:sanofi|data scientist',
  'company': 'Sanofi',
  'job_title': 'Data Scientist',
  'job_id': None,
  'source': 'Sanofi Workday <Sanofi@myworkday.com>',
  'applied_date': None,
  'status': 'application_confirmation',
  'last_update_date': '2026-04-01T14:04:12Z',
  'last_email_message_id': '19d495be58ba3445',
  'last_email_date_raw': 'Wed, 1 Apr 2026 14:04:12 +0000',
  'confidence': 0.85,
  'notes': 'The message indicates the application has been received and is currently under review by the Talent Acquisition Team.',
  'created_at': '2026-05-08T02:18:06.803709',
  'updated_at': '2026-05-08T02:18:06.803709'},
 {'id': 190,
  'app_key': 'company_title:sanofi|senior scientist - agentic ai systems - vaccines',
  'company': 'Sanofi',
  'job_title': 'Senior Scientist - Agentic AI Systems - Vaccines',
  'job_id': None,
  'source': 'Sanofi Workday <Sanofi@myworkday.com>',
  'applied_date': None,
  'status': 'application_confirmation',
  'last_update_date':

In [76]:
def update_application_status(app_key: str, new_status: str, note: str | None = None):
    now = datetime.utcnow().isoformat()
    conn = sqlite3.connect(DB_PATH)
    try:
        conn.execute(
            """
            UPDATE applications
            SET status = ?, notes = COALESCE(?, notes), updated_at = ?
            WHERE app_key = ?;
            """,
            (new_status, note, now, app_key),
        )

        conn.execute(
            """
            INSERT INTO app_events (app_key, event_type, event_date, gmail_message_id, raw_json, created_at)
            VALUES (?, ?, ?, NULL, ?, ?);
            """,
            (
                app_key,
                f"manual:{new_status}",
                now,
                json.dumps({"note": note, "status": new_status}, ensure_ascii=False),
                now,
            ),
        )

        conn.commit()
    finally:
        conn.close()


# How to use:
# 1) Copy an app_key from the apps list cell above
# 2) Run something like:
# update_application_status("job_id:12345", "rejection", "Got rejection email on May 7")

In [77]:
def get_application_timeline(app_key: str):
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    try:
        rows = conn.execute(
            """
            SELECT event_type, event_date, gmail_message_id, created_at
            FROM app_events
            WHERE app_key = ?
            ORDER BY created_at ASC;
            """,
            (app_key,),
        ).fetchall()
        return [dict(r) for r in rows]
    finally:
        conn.close()


# Example:
# timeline = get_application_timeline("job_id:12345")
# timeline

In [78]:
# Build a Pandas view of applications (applied + rejection if available)
# with a FAISS step to show semantic matching using TF-IDF vectors.

# %pip -q install faiss-cpu pandas scikit-learn

import pandas as pd
import numpy as np
import sqlite3

import faiss
from sklearn.feature_extraction.text import TfidfVectorizer

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
try:
    apps = conn.execute(
        """
        SELECT app_key, company, job_title, job_id, applied_date, status, updated_at
        FROM applications
        ORDER BY updated_at DESC;
        """
    ).fetchall()

    if not apps:
        raise RuntimeError("No rows in applications table yet. Run process_inbox_to_db() first.")

    apps_df = pd.DataFrame([dict(r) for r in apps])

    # Grab event dates for confirmations and rejections.
    events = conn.execute(
        """
        SELECT app_key, event_type, event_date
        FROM app_events
        WHERE event_type IN ('application_confirmation', 'rejection', 'interview')
        ;
        """
    ).fetchall()

finally:
    conn.close()

events_df = pd.DataFrame([dict(r) for r in events])

# applied_date: earliest application_confirmation event_date (fallback to applications.applied_date)
apply_dates = (
    events_df[events_df["event_type"] == "application_confirmation"]
    .groupby("app_key")["event_date"]
    .min()
    .rename("date_of_applied")
    .reset_index()
)

rejection_dates = (
    events_df[events_df["event_type"] == "rejection"]
    .groupby("app_key")["event_date"]
    .min()
    .rename("date_of_rejection")
    .reset_index()
)

out = apps_df.merge(apply_dates, on="app_key", how="left").merge(rejection_dates, on="app_key", how="left")

# Status rule: if a rejection exists, mark as rejection; else keep existing status.
out["status"] = np.where(out["date_of_rejection"].notna(), "rejection", out["status"])

# FAISS semantic matching step (TF-IDF vectors)
# We embed (company + job_title) and do nearest-neighbor search; then we keep the row mapping.
# This demonstrates FAISS usage, but the final applied/rejection dates come from SQL events.
texts = (out["company"].fillna("") + " " + out["job_title"].fillna("")).tolist()
vectorizer = TfidfVectorizer(max_features=4000, ngram_range=(1, 2))
X = vectorizer.fit_transform(texts).astype(np.float32)

# Convert sparse matrix to dense for FAISS (fine for small datasets; for large scale use a different approach)
X_dense = X.toarray().astype(np.float32)

index = faiss.IndexFlatL2(X_dense.shape[1])
index.add(X_dense)

# Example: find top-1 similar record for each row (distance computed in embedding space)
D, I = index.search(X_dense, 1)
# I should generally point to itself for the same row, but we keep it to show the pipeline.
out["faiss_self_match_idx"] = I[:, 0]
out["faiss_self_distance"] = D[:, 0]

# Final display columns
final_cols = [
    "company",
    "job_title",
    "date_of_applied",
    "date_of_rejection",
    "status",
]

# If date_of_applied is missing, try applications.applied_date
out["date_of_applied"] = out["date_of_applied"].fillna(out["applied_date"])

df = out[final_cols].sort_values(by="date_of_applied", ascending=False, na_position="last")

df

,company,job_title,date_of_applied,date_of_rejection,status
153,Waabi,"Research Engineer, Sensor Signal Processing",2026-05-07T19:34:02+00:00,NaN,application_confirmation
152,Xanadu Quantum Technologies,AI Specialist - AI For Research,2026-05-07T17:11:55Z,NaN,application_confirmation
150,RBC,Staff Data/AI Engineer,2026-05-07T16:23:49+00:00,NaN,application_confirmation
145,NaN,Machine Learning Bioinformatics Engineer,2026-05-04T14:50:05Z,NaN,application_confirmation
144,Electronic Arts,Data Scientist - Mobile Growth,2026-05-04T13:46:11Z,NaN,application_confirmation
...,...,...,...,...,...
146,Cantire,Senior Data Scientist,NaN,NaN,interview
147,Stripe,"2025-2026 PhD Intern, Data Scientist",NaN,NaN,interview
148,"Intuition Machines, Inc.",Senior/Lead ML Applied Scientist,NaN,2026-05-06T09:06:10Z,rejection
149,Copoly.ai,Machine Learning Bioinformatics Engineer,NaN,2026-05-07T15:40:05Z,rejection


In [79]:
df_out = df.copy()
df_out["date_of_applied"] = pd.to_datetime(df_out["date_of_applied"], utc=True).dt.strftime("%Y-%m-%d")
df_out["date_of_rejection"] = pd.to_datetime(df_out["date_of_rejection"], utc=True).dt.strftime("%Y-%m-%d")
df_out = df_out.sort_values(by="date_of_applied", ascending=True, na_position="last").reset_index(drop=True)
df_out.to_csv("applications_with_dates.csv", index=False)